<a href="https://colab.research.google.com/github/RicardoPaizinho/HardwareTestApp/blob/master/PokeGeminiMonteCarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
import os
import os
import sys
import json
import glob
import shutil
import stat
import tarfile
import random
import math
import traceback
import subprocess
from dataclasses import dataclass, field
from collections import Counter
from typing import Optional

# 1. Montar o Google Drive
drive.mount('/content/drive')
from google.colab import drive


# 2. Caminho da sua pasta no Drive
drive_path = '/content/drive/MyDrive/Pokemon_TCG_AI'
os.makedirs(drive_path, exist_ok=True) # Cria a pasta se não existir

# 3. Configurar a chave da API do Kaggle buscando da sua pasta
kaggle_dir = '/root/.kaggle'
os.makedirs(kaggle_dir, exist_ok=True)
!cp {drive_path}/kaggle.json {kaggle_dir}/kaggle.json
!chmod 600 {kaggle_dir}/kaggle.json

# 4. Baixar e extrair o simulador oficial (Simulation Division)
!pip install -q kaggle
print("Baixando o simulador do Kaggle...")
!kaggle competitions download -c pokemon-tcg-ai-battle
!unzip -q pokemon-tcg-ai-battle.zip -d ptcg_engine
print("Download e extração concluídos!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Baixando o simulador do Kaggle...
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
unzip:  cannot find or open pokemon-tcg-ai-battle.zip, pokemon-tcg-ai-battle.zip.zip or pokemon-tcg-ai-battle.zip.ZIP.
Download e extração concluídos!


Célula 2: Mapear a API e Instalar o Stable-Baselines3

In [16]:
from google.colab import drive
import sys
import os
import glob

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Procurar a pasta do motor 'cg' dentro do seu Drive nas pastas que você mencionou
print("Buscando o motor do jogo no Google Drive...")
search_paths = [
    '/content/drive/MyDrive/ptcg_data/**/cg',
    '/content/drive/MyDrive/ptcgProgram 22/**/cg',
    '/content/drive/MyDrive/Pokemon_TCG_AI/**/cg'
]

cg_path = []
for path in search_paths:
    cg_path = glob.glob(path, recursive=True)
    if len(cg_path) > 0:
        break

if len(cg_path) > 0:
    # Pega o diretório PAI da pasta 'cg' e adiciona ao sistema Python
    engine_dir = os.path.dirname(cg_path[0])
    if engine_dir not in sys.path:
        sys.path.append(engine_dir)
    print(f"✅ Motor do jogo mapeado com sucesso direto do Drive em: {engine_dir}")
else:
    print("❌ Erro: Motor 'cg' não encontrado. Verifique se a pasta com os arquivos C++ e Python realmente está no Drive.")

# 3. Testar as importações da API Oficial e instalar bibliotecas da IA
!pip install -q stable-baselines3[extra] sb3-contrib

try:
    from cg.game import battle_start, battle_finish, battle_select
    from cg.api import to_observation_class
    print("✅ Importações rodando perfeitamente! Pode seguir para o treinamento.")
except ModuleNotFoundError as e:
    print(f"❌ Erro de importação: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Buscando o motor do jogo no Google Drive...
✅ Motor do jogo mapeado com sucesso direto do Drive em: /content/drive/MyDrive/ptcg_data/sample_submission/sample_submission
✅ Importações rodando perfeitamente! Pode seguir para o treinamento.


main------------

Célula 3: O Ambiente Gym e o Treinamento Base (MaskablePPO)

In [17]:
import os
import math
import random
import torch
import torch.nn
import torch.nn.functional

from cg.api import (
    AreaType, Card, Observation, OptionType, PlayerState, Pokemon,
    SearchState, SelectContext, all_attack, all_card_data,
    search_begin, search_end, search_step, to_observation_class,
)

# --- Constantes e Tabelas de Pesquisa da API ---
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}
card_count = max(all_card, key=lambda c: c.cardId).cardId + 1

attack_count = max(all_attack(), key=lambda a: a.attackId).attackId + 1

num_words_encoder = 24
encoder_size = 22000

decoder_main_feature = 8
decoder_attack_offset = 14
decoder_card_offset = decoder_attack_offset + attack_count
decoder_size = decoder_card_offset + (1 + decoder_main_feature + SelectContext.RECOVER_SPECIAL_CONDITION) * card_count

SEARCH_COUNT = 10 # Limite seguro para evitar Timeout no Kaggle

# --- Estruturas da Rede Neural ---
class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_feedforward: int):
        super(DecoderLayer, self).__init__()
        self.attention = torch.nn.MultiheadAttention(d_model, num_heads)
        self.fc1 = torch.nn.Linear(d_model, d_feedforward)
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        y, _ = self.attention(x, encoder_out, encoder_out, need_weights=False)
        res = self.norm1(x + y)
        y = self.fc1(res)
        y = torch.nn.functional.relu(y)
        y = self.fc2(y)
        return self.norm2(res + y)

class MyModel(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_feedforward: int, num_layers_encoder: int, num_layers_decoder: int):
        super(MyModel, self).__init__()
        self.d_model = d_model
        self.encoder_bag = torch.nn.EmbeddingBag(encoder_size, d_model, mode="sum")
        encoder_layer = torch.nn.TransformerEncoderLayer(d_model, num_heads, d_feedforward, 0)
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers_encoder, enable_nested_tensor=False)
        self.encoder_fc = torch.nn.Linear(d_model, 1)
        self.decoder_bag = torch.nn.EmbeddingBag(decoder_size, d_model, mode="sum")
        self.decoder = torch.nn.ModuleList()
        for _ in range(num_layers_decoder):
            self.decoder.append(DecoderLayer(d_model, num_heads, d_feedforward))
        self.decoder_fc = torch.nn.Linear(d_model, 1)

    def forward(self, index_encoder: torch.Tensor, value_encoder: torch.Tensor, offset_encoder: torch.Tensor,
                index_decoder: torch.Tensor, value_decoder: torch.Tensor, offset_decoder: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        v = self.encoder_bag(index_encoder, offset_encoder, value_encoder)
        v = v.reshape(-1, num_words_encoder, self.d_model).transpose(0, 1)
        batch_size = v.size(1)
        encoder_out = self.encoder(v)
        v = self.encoder_fc(encoder_out)
        v = torch.tanh(v.mean(0))
        p = self.decoder_bag(index_decoder, offset_decoder, value_decoder)
        p = p.reshape(batch_size, -1, self.d_model).transpose(0, 1)
        for layer in self.decoder:
            p = layer(p, encoder_out)
        p = self.decoder_fc(p)
        p = p.transpose(0, 1).view(batch_size, -1)
        p = torch.tanh(p)
        return (v, p)

# --- Classes Auxiliares MCTS e Features ---
class SparseVector:
    def __init__(self):
        self.index = []
        self.value = []
        self.offset = []
        self.pos = 0

    def add(self, index: int, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos + index)
            self.value.append(value)

    def add_pos(self, pos: int):
        self.pos += pos

    def add_single(self, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos)
            self.value.append(value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))

class LearnSample:
    def __init__(self, value: float, policy: list[float], sv_enc: SparseVector, sv_dec:SparseVector):
        self.value = value
        self.policy = policy
        self.sv_enc = sv_enc
        self.sv_dec = sv_dec

class Child:
    def __init__(self, select: list[int], prob: float):
        self.node = None
        self.select = select
        self.prob = prob

class Node:
    def __init__(self, parent: 'Node | None', state: SearchState):
        self.value = -2.0
        self.total = 0.0
        self.visit = 0
        self.parent = parent
        self.children = []
        self.state = state

    def backprop(self, value: float):
        self.total += value
        self.visit += 1
        if self.parent != None:
            self.parent.backprop(value)

def add_card(sv: SparseVector, card: Card | Pokemon | None):
    if card != None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)

def add_cards(sv: SparseVector, cards: list[Card] | None, value: float):
    if cards != None:
        for card in cards:
            sv.add(card.id, value)
    sv.add_pos(card_count)

def add_pokemon(sv: SparseVector, poke: Pokemon | None):
    if poke == None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
    else:
        sv.add_single(0)
        sv.add_single(poke.hp / 400)
        add_card(sv, poke)
        add_cards(sv, poke.tools, 1.0)
        add_cards(sv, poke.energyCards, 0.5)

def add_player(sv: SparseVector, ps: PlayerState):
    sv.add_single(ps.deckCount / 60)
    sv.add_single(len(ps.discard) / 60)
    sv.add_single(ps.handCount / 8)
    sv.add_single(len(ps.bench) / 5)
    sv.add(len(ps.prize), 1)
    sv.add_pos(7)
    sv.add_single(ps.poisoned)
    sv.add_single(ps.burned)
    sv.add_single(ps.asleep)
    sv.add_single(ps.paralyzed)
    sv.add_single(ps.confused)
    add_cards(sv, ps.discard, 0.25)

def get_encoder_input(obs: Observation, your_deck: list[int]) -> SparseVector:
    your_index = obs.current.yourIndex
    state = obs.current
    sv = SparseVector()
    for i in range(2):
        ps = state.players[i ^ your_index]
        for j in range(8):
            sv.word_start()
            pos = sv.pos
            if j < len(ps.bench):
                add_pokemon(sv, ps.bench[j])
            else:
                add_pokemon(sv, None)
            if j != 7:
                sv.pos = pos
    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        if 0 < len(ps.active):
            add_pokemon(sv, ps.active[0])
        else:
            add_pokemon(sv, None)
    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        add_player(sv, ps)

    sv.word_start()
    add_cards(sv, state.players[your_index].hand, 0.25)

    sv.word_start()
    for id in your_deck:
        sv.add(id, 0.25)
    sv.add_pos(card_count)

    sv.word_start()
    add_cards(sv, state.stadium, 1.0)

    sv.word_start()
    sv.add_single(1)
    sv.add_single(state.turn / 10)
    sv.add_single(state.firstPlayer == your_index)
    return sv

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK: return obs.select.deck[index]
        case AreaType.HAND: return ps.hand[index]
        case AreaType.DISCARD: return ps.discard[index]
        case AreaType.ACTIVE: return ps.active[index]
        case AreaType.BENCH: return ps.bench[index]
        case AreaType.PRIZE: return ps.prize[index]
        case AreaType.STADIUM: return obs.current.stadium[index]
        case AreaType.LOOKING: return obs.current.looking[index]
        case _: return None

def decoder_main(sv: SparseVector, feature_index: int, card: Card | Pokemon | None):
    if card != None:
        sv.add(decoder_card_offset + feature_index * card_count + card.id, 1)

def decoder_card_id(sv: SparseVector, context: SelectContext, card_id: int):
    sv.add(decoder_card_offset + (decoder_main_feature + context) * card_count + card_id, 1)

def decoder_card(sv: SparseVector, context: SelectContext, card: Card | Pokemon | None):
    if card != None:
        decoder_card_id(sv, context, card.id)

def get_decoder_input(obs: Observation, actions: list[list[int]]) -> SparseVector:
    sv = SparseVector()
    your_index = obs.current.yourIndex
    ps = obs.current.players[your_index]
    context = obs.select.context
    for action in actions:
        sv.word_start()
        if len(action) == 0:
            sv.add(0, 1)
            continue
        for i in action:
            o = obs.select.option[i]
            match o.type:
                case OptionType.END: sv.add(1, 1)
                case OptionType.YES: sv.add(2, 1)
                case OptionType.NO: sv.add(3, 1)
                case OptionType.SPECIAL_CONDITION: sv.add(4 + o.specialConditionType, 1)
                case OptionType.NUMBER: sv.add(9 + min(o.number, 4), 1)
                case OptionType.ATTACK: sv.add(decoder_attack_offset + o.attackId, 1)
                case OptionType.PLAY: decoder_main(sv, 0, ps.hand[o.index])
                case OptionType.ATTACH:
                    decoder_main(sv, 1, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 2, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.EVOLVE:
                    decoder_main(sv, 3, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 4, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.ABILITY: decoder_main(sv, 5, get_card(obs, o.area, o.index, your_index))
                case OptionType.DISCARD: decoder_main(sv, 6, get_card(obs, o.area, o.index, your_index))
                case OptionType.RETREAT: decoder_main(sv, 7, ps.active[0])
                case OptionType.CARD: decoder_card(sv, context, get_card(obs, o.area, o.index, o.playerIndex))
                case OptionType.TOOL_CARD:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.tools[o.toolIndex])
                case OptionType.ENERGY_CARD | OptionType.ENERGY:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.energyCards[o.energyIndex])
                case OptionType.SKILL: decoder_card_id(sv, context, o.cardId)
    return sv

def eval_nn(sv_enc: SparseVector, sv_dec:SparseVector, model: MyModel) -> tuple[float, list[float]]:
    device = next(model.parameters()).device
    value, policy = model(
        torch.tensor(sv_enc.index, dtype=torch.int32, device=device),
        torch.tensor(sv_enc.value, dtype=torch.float32, device=device),
        torch.tensor(sv_enc.offset, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.index, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.value, dtype=torch.float32, device=device),
        torch.tensor(sv_dec.offset, dtype=torch.int32, device=device))
    return (value.tolist()[0][0], policy.tolist()[0])

def create_node(parent: Node | None, search_state: SearchState, your_index: int, your_deck: list[int], model: MyModel) -> tuple[Node, LearnSample | None]:
    node = Node(parent, search_state)
    obs = search_state.observation
    state = obs.current
    if state.result >= 0:
        if state.result == 2:
            node.value = 0
        elif state.result == your_index:
            node.value = 1
        else:
            node.value = -1
        node.backprop(node.value)
        sample = None
    else:
        actions = []
        indices = list(range(obs.select.maxCount))
        for _ in range(64):
            actions.append(indices.copy())
            for i in range(len(indices)):
                index = len(indices) - i - 1
                if indices[index] < len(obs.select.option) - i - 1:
                    indices[index] += 1
                    for j in range(index+1, len(indices)):
                        indices[j] = indices[j - 1] + 1
                    break
            else:
                break

        sv_enc = get_encoder_input(obs, your_deck)
        sv_dec = get_decoder_input(obs, actions)
        value, policy = eval_nn(sv_enc, sv_dec, model)
        v = value
        if state.yourIndex != your_index:
            v = -v
        node.value = v
        node.backprop(v)

        sum_policy = 0.0
        for i in range(len(policy)):
            p = math.exp(policy[i] * 10.0)
            node.children.append(Child(actions[i], p))
            sum_policy += p
        for c in node.children:
            c.prob /= sum_policy
        sample = LearnSample(value, policy, sv_enc, sv_dec)
    return (node, sample)

def mcts_agent(obs_dict: dict, your_deck: list[int], model: MyModel) -> list[int]:
    obs = to_observation_class(obs_dict)

    if obs.current is None or not hasattr(obs, 'select') or obs.select is None or not obs.select.option:
        return []

    your_index = obs.current.yourIndex
    state = obs.current
    active = state.players[1 - your_index].active if state.players and len(state.players) > 1 else []

    search_state = search_begin(
        obs,
        your_deck=random.sample(your_deck, min(state.players[your_index].deckCount, len(your_deck))),
        your_prize=random.sample(your_deck, min(len(state.players[your_index].prize), len(your_deck))),
        opponent_deck=[1072] * state.players[1 - your_index].deckCount,
        opponent_prize=[1] * len(state.players[1 - your_index].prize),
        opponent_hand=[1] * state.players[1 - your_index].handCount,
        opponent_active=[1072] if len(active) > 0 and active[0] == None else [])

    root, _ = create_node(None, search_state, your_index, your_deck, model)

    if not root.children:
        search_end()
        return []

    for _ in range(SEARCH_COUNT):
        current = root
        while True:
            value = -1e9
            c = 0.4 * math.sqrt(current.visit)
            next_node = None
            for child in current.children:
                visit = 0
                if child.node == None:
                    v = current.total / current.visit if current.visit > 0 else 0
                else:
                    v = child.node.total / child.node.visit
                    visit = child.node.visit

                if current.state.observation.current.yourIndex != your_index:
                    v = -v
                v += c * child.prob / (1 + visit)
                if value < v:
                    value = v
                    next_node = child

            if next_node is None:
                break

            if next_node.node == None:
                new_search_state = search_step(current.state.searchId, next_node.select)
                next_node.node, _ = create_node(current, new_search_state, your_index, your_deck, model)
                break
            else:
                current = next_node.node
                if current.state.observation.current.result >= 0:
                    current.backprop(current.value)
                    break

    max_child = None
    max_visit = -1
    for child in root.children:
        if child.node != None:
            if max_visit < child.node.visit:
                max_child = child
                max_visit = child.node.visit

    search_end()

    if max_child is not None and max_child.select is not None:
        return max_child.select
    elif root.children:
        return root.children[0].select
    return []

# --- Inicialização Global para Inferência ---
device = torch.device("cpu")

model = MyModel(256, 8, 512, 2, 2).to(device)

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

model_path = os.path.join(current_dir, 'model4.pth')
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# Leitura rigorosa do deck.csv garantindo exatamente 60 cartas
deck_path = os.path.join(current_dir, 'deck.csv')
my_deck = []
if os.path.exists(deck_path):
    with open(deck_path, 'r') as f:
        for line in f.readlines():
            line = line.strip()
            if line.isdigit():
                my_deck.append(int(line))

if len(my_deck) != 60:
    if len(my_deck) > 60:
        my_deck = my_deck[:60]
    elif len(my_deck) > 0:
        my_deck.extend([my_deck[-1]] * (60 - len(my_deck)))
    else:
        my_deck = [1072] * 60

def agent(obs_dict: dict) -> list[int]:
    try:
        with torch.inference_mode():
            obs = to_observation_class(obs_dict)

            if obs.current is None or not hasattr(obs, 'select') or obs.select is None:
                return [0] if obs.select and obs.select.option and len(obs.select.option) > 0 else []

            options = obs.select.option
            max_count = obs.select.maxCount

            if not options:
                return []

            action = mcts_agent(obs_dict, my_deck, model)

            if action is not None:
                valid_action = [idx for idx in action if isinstance(idx, int) and 0 <= idx < len(options)]
                if len(valid_action) > 0:
                    return valid_action[:max_count]

            return [0] if max_count > 0 and len(options) > 0 else []

    except Exception:
        try:
            if 'obs' in locals() and hasattr(obs, 'select') and obs.select and obs.select.option:
                return [0]
        except:
            pass
        return []

In [7]:
import os
import random
import math
import torch
import torch.nn as nn
import torch.optim as optim

# Importações do motor do jogo (certifique-se de que a Célula 2 de mapeamento já rodou)
from cg.game import battle_start, battle_finish, battle_select
from cg.api import to_observation_class

# 1. Configuração do Dispositivo (Usa GPU T4 do Colab se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Usando dispositivo para treino: {device}")

# ==========================================
# 2. NOVA ARQUITETURA REDUZIDA E OTIMIZADA
# ==========================================
d_model = 256
num_heads = 8
d_feedforward = 512
num_layers_encoder = 2
num_layers_decoder = 2

# Certifique-se de que a classe MyModel está definida no seu ambiente (como nos códigos anteriores)
model = MyModel(d_model, num_heads, d_feedforward, num_layers_encoder, num_layers_decoder).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Funções de perda
criterion_value = nn.MSELoss()
criterion_policy = nn.MSELoss()

# Defina seus decks para o treinamento cruzado (Multi-Deck)
ZACIAN_DECK = [
    721,721,722,722,722,722,723,723,723,723,
    1092,1121,1121,1145,1145,1163,1163,
    1219,1219,1219,1219,1227,1227,1227,1227,
    1262,1262,
    3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
]
NOVO_DECK = [
1086,1086,1086,1086,1227,1227,1227,1227,
1219,1219,1219,1219,112,112,112,112,
1259,1259,1259,1259,7,7,7,7,7,7,7,7,7,7,
1152,1152,1152,1152,647,647,647,1097,1097,1097,
646,646,646,646,104,104,648,648,648,1079,1079,1079,
860,860,1122,1080,1137,1182,1182,1231
]
# Você pode adicionar mais decks aqui no pool se quiser expandir a generalização:
DECK_POOL = [ZACIAN_DECK, NOVO_DECK]

print("Iniciando treinamento com o modelo leve otimizado...")

NUM_EPISODES = 50  # Ajuste para a quantidade de gerações desejada
model.train()

for episode in range(NUM_EPISODES):
    # Sorteia os decks para o auto-confronto
    deck_p0 = random.choice(DECK_POOL)
    deck_p1 = random.choice(DECK_POOL)

    obs_dict, start_data = battle_start(deck_p0, deck_p1)
    if start_data.errorPlayer >= 0:
        continue

    obs = to_observation_class(obs_dict)
    episode_samples = []
    done = False

    while not done:
        your_index = obs.current.yourIndex
        current_deck = deck_p0 if your_index == 0 else deck_p1

        if not obs.select or not obs.select.option:
            break

        actions = []
        indices = list(range(min(obs.select.maxCount, len(obs.select.option))))
        for _ in range(64):
            actions.append(indices.copy())
            for i in range(len(indices)):
                idx = len(indices) - i - 1
                if indices[idx] < len(obs.select.option) - i - 1:
                    indices[idx] += 1
                    for j in range(idx + 1, len(indices)):
                        indices[j] = indices[j - 1] + 1
                    break
            else:
                break

        if len(actions) == 0:
            break

        sv_enc = get_encoder_input(obs, current_deck)
        sv_dec = get_decoder_input(obs, actions)

        chosen_idx = random.randint(0, len(actions) - 1)

        obs_dict = battle_select(actions[chosen_idx])
        obs = to_observation_class(obs_dict)

        result = obs.current.result
        if result >= 0:
            done = True
            battle_finish()
            if result == 2:
                final_val = 0.0
            elif result == your_index:
                final_val = 1.0
            else:
                final_val = -1.0
        else:
            final_val = 0.0

        episode_samples.append((sv_enc, sv_dec, final_val, chosen_idx, len(actions)))

    # Loop de otimização por gradiente
    total_loss = 0
    for sv_enc, sv_dec, target_val, chosen_idx, num_acts in episode_samples:
        optimizer.zero_grad()

        t_index_enc = torch.tensor(sv_enc.index, dtype=torch.int32, device=device)
        t_value_enc = torch.tensor(sv_enc.value, dtype=torch.float32, device=device)
        t_offset_enc = torch.tensor(sv_enc.offset, dtype=torch.int32, device=device)

        t_index_dec = torch.tensor(sv_dec.index, dtype=torch.int32, device=device)
        t_value_dec = torch.tensor(sv_dec.value, dtype=torch.float32, device=device)
        t_offset_dec = torch.tensor(sv_dec.offset, dtype=torch.int32, device=device)

        pred_val, pred_policy = model(
            t_index_enc, t_value_enc, t_offset_enc,
            t_index_dec, t_value_dec, t_offset_dec
        )

        target_v_tensor = torch.tensor([[target_val]], dtype=torch.float32, device=device)
        target_p = torch.zeros_like(pred_policy)
        if chosen_idx < target_p.shape[1]:
            target_p[0, chosen_idx] = 1.0

        loss_v = criterion_value(pred_val, target_v_tensor)
        loss_p = criterion_policy(pred_policy, target_p)

        loss = loss_v + loss_p
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (episode + 1) % 10 == 0:
        print(f"Episode {episode+1}/{NUM_EPISODES} finalizado. Loss acumulada: {total_loss:.4f}")

# 3. Salva o novo modelo leve otimizado no Google Drive
save_dir = '/content/drive/MyDrive/Pokemon_TCG_AI'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'model4.pth')

torch.save(model.state_dict(), model_path)
print(f"✅ Modelo leve treinado e salvo com sucesso em: {model_path}")

🔥 Usando dispositivo para treino: cuda
Iniciando treinamento com o modelo leve otimizado...


KeyboardInterrupt: 

In [18]:
import tarfile
import os

# Defina o diretório onde estão seus arquivos no Drive
work_dir = '/content/drive/MyDrive/Pokemon_TCG_AI/'
os.chdir(work_dir)

# Cria o tar.gz incluindo o main.py, model4.pth, deck.csv E a pasta cg/
with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py')
    tar.add('model4.pth')
    tar.add('deck.csv')
    if os.path.exists('cg'):
        tar.add('cg')

print("Pacote submission.tar.gz criado com sucesso incluindo a pasta cg!")

Pacote submission.tar.gz criado com sucesso incluindo a pasta cg!
